In [ ]:
import numpy as np
import mascon_fusion as mf

## Load observations

In [ ]:
# load CRI filtered mascon solutions
filepath = '/Users/u6621369/python_codes/CRI_mascons/outputs/'
cri_string = 'CRI_ne50m_IMBIE3_E6_copy.nc'

filenames = {'JPL': filepath+'GRCTellus.JPL.200204_202601.GLO.RL06.3M.MSCNv04_'+cri_string,
             'GSFC': filepath+'gsfc.glb_.200204_202511_rl06v2.0_obp-ice6gd_'+cri_string,
             'CSR': filepath+'CSR_GRACE_GRACE-FO_RL0603_Mascons_all-corrections_'+cri_string,
             'ANU': filepath+'ANU_mascons_RL02_2002-2023_grid_'+cri_string
}

sJPL, sJPL.mascons = mf.load_mascon_grid(filenames['JPL'], load_solution=True)
sGSFC, sGSFC.mascons = mf.load_mascon_grid(filenames['GSFC'], load_solution=True)
sCSR, sCSR.mascons = mf.load_mascon_grid(filenames['CSR'], load_solution=True)
sANU, sANU.mascons = mf.load_mascon_grid(filenames['ANU'], load_solution=True)

# set up the inputs dictionary
inputs = {'JPL': sJPL, 'GSFC': sGSFC, 'CSR': sCSR, 'ANU': sANU}

In [ ]:
# print epoch range and number of mascons for each product
for name, grid in inputs.items():
    print(f"{name.ljust(5)}: {grid.decyear[0]:.2f} - {grid.decyear[-1]:.2f}, nmascons={len(grid.mascons['lon'])}")

## 2. Set up target

In [ ]:
# load target grid
target, target.mascons = mf.load_mascon_grid(filenames['GSFC'], load_solution=False)

In [ ]:
# create the target decyear array
target.decyear = mf.build_decyear_array(start_year=2002, end_year=2025, day=15)
ntimes = len(target.decyear)

## 3. Process observations

In [ ]:
# remove the mean field and interpolate to mid-month
for name, grid in inputs.items():
    print(f"Processing {name}...")
    # make sure mean is removed from all solutions
    grid.mascons['ewh'], grid.mascons['mean'] = mf.remove_mean_epoch(grid.mascons['ewh'], grid.decyear)
    grid.mean = grid.mascons['mean'][grid.mascon_id - 1]
    # then interpolate to the target decyear array
    grid.mascons['ewh'] = mf.interpolate_timeseries(grid.decyear, grid.mascons['ewh'], target.decyear, 
                                           max_gap=0.7/12, kind="linear", fill_value=np.nan)
    grid.decyear = target.decyear
    grid.ewh = np.zeros((len(grid.decyear), target.nlat, target.nlon))
    for t in range(len(grid.decyear)):
        grid.ewh[t] = grid.mascons['ewh'][t][grid.mascon_id - 1]

## 4. Set up fusion

In [ ]:
# build the design matrix
A, row_slices = mf.build_design_matrix(target, inputs)
nobs, nparams = A.shape[0], A.shape[1]

In [ ]:
# define experiment weighting
exp = {"id":"test", 
       "weights":{"GSFC":1.0,"CSR":1.0,"ANU":1.25,"JPL":1.5},
       "use_area":True
       }
# create the weight vector
W_vec = mf.build_weights(inputs, nobs, row_slices, exp["weights"], exp["use_area"])


In [ ]:
# plot eigen diagnostics
mf.plot_eigen_diagnostics(A, W_vec, row_slices, exp["id"])

In [ ]:
# run this once
solver, AtW, N_inv_diag = mf.fusion_setup(A, W_vec)

## 5. Solve fusion 

In [ ]:
# --------------------------------------------------
# Preallocate output arrays
# --------------------------------------------------

X      = np.full((ntimes, nparams), np.nan)
SE_all = np.full((ntimes, nparams), np.nan)
sigma0 = np.full(ntimes, np.nan)

block_res = {
    name: np.full((ntimes, inputs[name].nmascons), np.nan)
    for name in row_slices.keys()
}

block_rms_all = {
    name: np.full(ntimes, np.nan)
    for name in row_slices.keys()
}

# --------------------------------------------------
# Loop over epochs
# --------------------------------------------------

for t in range(ntimes):

    missing_sols = []
    sol_blocks = {}

    # --------------------------------------------------
    # Collect solutions and detect missing products
    # --------------------------------------------------

    for name, grid in inputs.items():

        sol = grid.mascons['ewh'][t]

        if np.any(np.isnan(sol)):
            missing_sols.append(name)
        else:
            sol_blocks[name] = sol

    # --------------------------------------------------
    # Case 1: full system available
    # --------------------------------------------------

    if not missing_sols:

        print(f"Epoch {t:3d}: all solutions present, solving full system")

        b_vec = np.concatenate([sol_blocks[name] for name in inputs.keys()])

        x_hat, SE, res, sigma0_sq = mf.fusion_solve(
            A, W_vec, solver, AtW, N_inv_diag, row_slices, b_vec
        )

        block_rms = mf.get_block_rms(res, W_vec, row_slices)

    # --------------------------------------------------
    # Case 2: products missing → skip
    # --------------------------------------------------

    else:

        missing_str = ", ".join(missing_sols)

        print(f"Epoch {t:3d}: missing {missing_str}, skipping")

        continue

    # --------------------------------------------------
    # Store outputs
    # --------------------------------------------------

    X[t, :]      = x_hat
    SE_all[t, :] = SE
    sigma0[t]    = sigma0_sq

    for name, rms in block_rms.items():
        sl = row_slices[name]
        block_res[name][t] = res[sl]
        block_rms_all[name][t] = rms


## 6. Save output

In [ ]:
# Fill in the target mascon objects
target.mascons['ewh'] = X
target.mascons['SE'] = SE_all
target.mascons['sigma0'] = sigma0

In [ ]:
# Build gridded EWH
target.ewh = target.mascons['ewh'][:, target.mascon_id - 1].reshape(
                   ntimes, target.nlat, target.nlon
        )
target.SE = target.mascons['SE'][:, target.mascon_id - 1].reshape(
                   ntimes, target.nlat, target.nlon
        )

In [ ]:
# Build vector and gridded residuals
target.mascons['block_rms'] = block_rms_all
target.mascons['residuals'] = {}
target.residuals = {}
for name, sl in row_slices.items():
    grid = inputs[name]
    # vector form
    target.mascons['residuals'][name] = block_res[name]
    # gridded form
    target.residuals[name] = block_res[name][:, grid.mascon_id - 1].reshape(
                     ntimes, target.nlat, target.nlon
        )

## 7. Analyse fusion

In [ ]:
mf.plot_fusion_diagnostics(target, sigma0, block_rms_all, SE_all, bins=100)

In [ ]:
mf.plot_solution_uncertainty(target, 2023.8)

In [ ]:
mf.plot_block_residuals(target, 2023.8)

## 8. Write fusion to file

In [ ]:
filename = f"../outputs/{cri_string.strip('.nc')}_MF_{exp['id']}_codetest.nc"
exp_name = cri_string.strip('.nc')+'_MF_'+exp['id']
print(f'Filename: {filename}')
print(f'Experiment Name: {exp_name}')

In [ ]:
mf.write_fusion_netcdf(
    filename, target, target.mascons, exp_name, 
    exp['weights'], exp['use_area'], block_norm,
)

In [ ]:
mf.write_fusion_residuals_netcdf(
    filename.replace('.nc', '_residuals.nc'), 
    target, target.mascons, inputs, exp_name
)